# Jour 3 — Stabiliser et piloter l’entraînement

## Objectifs

- distinguer capacité, optimisation et généralisation
- comprendre le rôle de Batch Normalization
- utiliser une réduction du learning rate et un arrêt anticipé
- lire ensemble loss, accuracy et évolution du learning rate


## Trois problèmes qui se ressemblent parfois

![Équilibre entre capacité et généralisation](../ressources/illustrations/jour_03/02_capacite_regularisation.png)

Un modèle peut être limité pour plusieurs raisons :

- sa **capacité** est insuffisante pour représenter la relation ;
- l’**optimisation** ne trouve pas de bons poids ;
- il apprend bien le train mais généralise mal.

Ajouter des neurones augmente la capacité, mais ne corrige pas automatiquement les données, le learning rate ou le surapprentissage.

On compare les courbes du train et de la validation. Deux scores faibles suggèrent souvent un sous-apprentissage. Un train très bon avec une validation nettement plus faible suggère plutôt un surapprentissage.


## Batch Normalization

![Principe de Batch Normalization](../ressources/illustrations/jour_03/02_batch_normalization.png)

`BatchNormalization` normalise les activations internes observées dans les batchs, puis apprend une remise à l’échelle et un décalage. Elle peut rendre l’optimisation plus stable et autoriser des progressions plus régulières.

Pendant l’entraînement, la couche utilise les statistiques des batchs et met à jour des moyennes mémorisées. Pendant l’inférence, elle utilise ces valeurs mémorisées. Le comportement est géré automatiquement par Keras.

Batch Normalization n’est pas un remplacement de la normalisation des entrées. Elle ne garantit pas non plus une meilleure généralisation dans toutes les architectures.


## Les callbacks pilotent la boucle

![ReduceLROnPlateau et EarlyStopping dans la boucle](../ressources/illustrations/jour_03/02_callbacks.png)

Un **callback** reçoit des informations pendant `fit` et peut réagir à la fin d’une époque.

- `ReduceLROnPlateau` réduit le learning rate lorsque `val_loss` ne progresse plus.
- `EarlyStopping` arrête après plusieurs époques sans amélioration.
- `restore_best_weights=True` restaure les meilleurs poids observés.

`patience` évite de réagir à une seule fluctuation. Les deux callbacks n’ont pas la même patience : on peut laisser le temps au learning rate réduit d’améliorer le modèle avant de l’arrêter.


### Ne pas confondre optimiseur, learning rate et callback

L’**optimiseur** définit la manière dont les gradients deviennent des corrections de poids. Adam conserve notamment des informations sur les gradients passés.

Le **learning rate** règle l’amplitude générale des corrections. Il appartient à l’optimiseur, mais un callback peut modifier sa valeur pendant `fit`.

Le callback n’apprend aucun poids et ne remplace pas l’optimiseur. Il observe l’historique puis déclenche une action prévue : réduire le pas, arrêter ou sauvegarder.

Une réduction trop précoce peut ralentir inutilement l’apprentissage. Une patience trop longue peut faire perdre du temps ou laisser le modèle surapprendre. Ces réglages restent des hyperparamètres à comparer sur la validation.


## Les outils de régularisation n’agissent pas au même endroit

![Régularisation et généralisation](../ressources/illustrations/jour_02/03_regularisation_generalisation.png)

L’augmentation agit sur les exemples présentés. Dropout agit sur les connexions utilisées pendant le train. Batch Normalization agit sur les activations. L’arrêt anticipé agit sur la durée de l’entraînement.

Empiler toutes les techniques sans diagnostic complique l’expérience. On modifie idéalement un choix à la fois, on conserve le même découpage et on compare les courbes ainsi que les erreurs concrètes.


## Construire un CNN stabilisé

La couche convolutionnelle utilise `use_bias=False` car Batch Normalization apprend déjà un décalage. Une activation ReLU séparée vient après la normalisation.

Une petite translation aléatoire enrichit le train. Elle est désactivée pendant la validation et l’inférence.

Le nombre d’époques est un maximum. Les callbacks peuvent réduire le learning rate ou arrêter plus tôt.


Cette démonstration assemble les techniques déjà décrites. `RandomTranslation(0.05, 0.05)` autorise de petits déplacements, jusqu'à environ 5 % de chaque dimension. `use_bias=False` retire le biais de la convolution car la normalisation qui suit apprend son propre décalage. `Dropout(0.25)` masque environ un quart des sorties pendant le train seulement.

Dans `ReduceLROnPlateau`, `factor=0.5` divise le learning rate par deux après deux époques sans amélioration ; `min_lr=1e-5` fixe la borne basse à `0.00001`. L'arrêt anticipé attend six époques, pour laisser une chance au pas réduit d'améliorer la validation.

Cette expérience montre comment assembler ces outils. Pour déterminer l'effet d'une technique particulière, il faudrait ensuite la comparer seule à une référence, avec le même découpage.


In [ ]:
# Réglages de l’environnement, avant le chargement de TensorFlow.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

# NumPy gère les tableaux et les calculs numériques.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

# Fixe le hasard pour faciliter les comparaisons.
tf.keras.utils.set_random_seed(42)
try:
    # Limite les ressources utilisées par cette petite démonstration.
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass

# Charge les images et leurs labels ensemble.
digits = load_digits()


In [ ]:
# Convertit les pixels en décimaux et les ramène à l’échelle [0, 1].
images = digits.images.astype("float32") / 16.0
# Conserve un chiffre entier comme bonne réponse par image.
labels = digits.target.astype("int64")
# Sépare ensemble les entrées et les réponses pour garder leur association.
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42
)
X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]


In [ ]:
# Les couches sont reliées dans l’ordre de cette liste.
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),
    tf.keras.layers.RandomTranslation(0.05, 0.05),
    tf.keras.layers.Conv2D(16, 3, padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(10, activation="softmax"),
])
# Associe le réseau, sa loss et l’optimiseur ; aucun poids ne change encore.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.002),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)


In [ ]:
# Réduit le pas quand la validation ne s’améliore plus.
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5,
)
# Suit la validation et restaure les meilleurs poids observés.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=6, restore_best_weights=True,
)

# Entraîne les poids sur le train et observe la validation séparément.
history = model.fit(
    X_train, y_train,
    validation_split=0.20,
    epochs=35,
    batch_size=32,
    callbacks=[reduce_lr, early_stop],
    verbose=0,
)
print("Époques effectuées :", len(history.history["loss"]))


## Lire plusieurs courbes ensemble

La loss sert à l’optimisation. L’accuracy indique seulement si la classe arrivée première est correcte. Il est donc possible que la loss s’améliore légèrement sans changement visible d’accuracy.

Keras enregistre le learning rate dans l’historique avec les autres mesures. Une baisse montre que `ReduceLROnPlateau` a réagi.

On cherche une progression cohérente de la validation, pas une courbe parfaitement lisse. Le meilleur résultat isolé peut provenir d’une fluctuation.


`pd.DataFrame(history.history)` range les mesures dans un tableau, une ligne par époque. La sélection `[["loss", "val_loss"]]` garde deux colonnes.

Le nom du learning rate peut être `learning_rate` ou `lr` selon les versions ; le code accepte les deux. L'échelle `log` rend visibles les changements multiplicatifs, comme une division par deux. `idxmin()` donne l'indice de la plus petite loss ; on ajoute 1 car les indices Python commencent à zéro et les époques affichées à un.


In [ ]:
# Une ligne de mesures par époque, dans un tableau lisible.
history_df = pd.DataFrame(history.history)
# Réserve les emplacements graphiques, sans modifier les données.
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

history_df[["loss", "val_loss"]].plot(ax=axes[0], title="Loss")
history_df[["accuracy", "val_accuracy"]].plot(ax=axes[1], title="Accuracy")

lr_column = "learning_rate" if "learning_rate" in history_df else "lr"
history_df[lr_column].plot(ax=axes[2], title="Learning rate")
axes[2].set_yscale("log")

for ax in axes:
    ax.set_xlabel("Époque")
plt.tight_layout()
# Affiche le résultat des calculs précédents.
plt.show()

# Mesure les nouvelles images sans corriger les poids.
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
# Passe de l’indice Python, qui commence à 0, au numéro d’époque.
best_epoch = int(history_df["val_loss"].idxmin() + 1)
print("Meilleure époque selon val_loss :", best_epoch)
print("Accuracy test :", round(test_accuracy, 3))


## Interpréter sans surpromettre

Si le learning rate baisse puis que `val_loss` recommence à progresser, la réduction du pas a été utile. Si rien ne change, le blocage peut venir de la capacité, des données ou d’un minimum difficile.

L’arrêt anticipé choisit les poids à partir de la validation. Le test doit donc rester séparé jusqu’à la fin.

Un score élevé sur `digits` ne garantit pas un comportement identique sur d’autres écritures ou résolutions. La stabilité de l’entraînement et la robustesse aux changements sont deux questions différentes.


## À vous de jouer — résumer l’entraînement

Calculez et affichez :

- le nombre d’époques réellement effectuées ;
- la meilleure `val_accuracy` ;
- l’époque correspondante ;
- le learning rate final.

Rédigez ensuite une phrase indiquant si le callback de réduction a été activé.

`len(history_df)` compte les époques. `max()` garde la meilleure mesure ; `idxmax()` retrouve sa ligne. `iloc[0]` lit la première ligne et `iloc[-1]` la dernière.


In [ ]:
# Extrayez les informations depuis history_df.
pass


## À retenir

Piloter un entraînement consiste à observer, comparer et modifier une hypothèse précise. Batch Normalization peut stabiliser les activations ; les callbacks ajustent la boucle ; la validation indique si ces choix améliorent la généralisation.
